# 02 - Positive control, time-matched controls, and capable VLMs

**Question:** does the reliance estimate R = S - N detect reliance when it exists, and do capable VLMs show it?

**Changes from notebook 01**
1. **Time-matched control.** Notebook 01's same-label partner was closer in time than the flip partner (mean gap 1.95 vs 3.55 follow-ups), which biases R upward for any model whose output drifts with time. The control is now a **same-label pair from the same patient at exactly the same signed follow-up gap** as the flip pair. In simulation this removed the bias (blind-but-drifting model: R = 0.018 with the old control, -0.004 with the new one).
2. **Positive control:** TorchXRayVision DenseNet trained on **CheXpert only** (not NIH, so no train/test leakage). It reads effusion, so it should give R > 0. If it does not, the design is wrong.
3. **Models:** Qwen2.5-VL-3B (pilot baseline), Qwen2.5-VL-7B, MedGemma-4B - all on the **same anchors**, so models can be compared with paired tests.
4. Estimates on the probability scale and the **log-odds scale** (comparable across models with different calibration), plus AUC to show whether each model can see the finding at all.

**Settings before running**
- *Add Input*: **NIH Chest X-rays** dataset (`nih-chest-xrays/data`, Datasets tab).
- *Accelerator*: **GPU T4 x2** (the 7B model and MedGemma in float32 need both GPUs). *Internet*: **On**.
- MedGemma is gated: accept its licence on Hugging Face (`google/medgemma-4b-it`) and add a Kaggle secret named **HF_TOKEN**. Without it, MedGemma is skipped and the rest still runs.

Expected runtime: roughly 1-2 hours (about 1,500 images per VLM).

In [1]:
!pip -q install torchxrayvision
# If loading Qwen2.5-VL or MedGemma fails with an unknown-architecture error, also run:
# !pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 49.4 MB/s eta 0:00:00


In [2]:
%%writefile pilot_lib.py
# ---- Pair construction and estimation (no model code) ----
import numpy as np
import pandas as pd

NIH_FINDINGS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion", "Emphysema",
                "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
                "Pneumonia", "Pneumothorax"]


def add_label(df, finding):
    out = df.copy()
    out["label"] = out["Finding Labels"].str.split("|").apply(lambda L: int(finding in L))
    return out


def build_anchor_triples(df, finding, same_view=True, max_anchors_per_patient=2, seed=0,
                         match_gap=False, gap_tol=1, control="anchor"):
    """For each anchor image A, find in the SAME patient:
       flip partner  F: another study with the opposite label for `finding` (should change the answer)
       same partner  S: another study with the same label (should not change the answer)
    Partners are the nearest follow-ups (smallest |follow-up gap|) and share A's view if same_view.
    Using a same-patient, different-day image as the control means both arms carry the same kind
    of nuisance change (positioning, exposure, time), so R = S - N subtracts it."""
    rng = np.random.default_rng(seed)
    d = add_label(df, finding)
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    d = d[d.groupby(key)["Image Index"].transform("size") >= 3]
    rows = []
    for k, g in d.groupby(key):
        pid = k[0] if isinstance(k, tuple) else k
        img = g["Image Index"].to_numpy(); lab = g["label"].to_numpy()
        fu = g["Follow-up #"].to_numpy(); view = g["View Position"].to_numpy()
        if lab.min() == lab.max():
            continue
        anchors = []
        for i in range(len(g)):
            other = np.arange(len(g)) != i
            fl = np.where(other & (lab != lab[i]))[0]
            sm = np.where(other & (lab == lab[i]))[0]
            if len(fl) == 0 or len(sm) == 0:
                continue
            f = fl[np.abs(fu[fl] - fu[i]).argmin()]
            if control == "pair":
                # control = a same-label PAIR (u, v) of this patient with EXACTLY the flip pair's signed
                # follow-up gap; prefer the anchor's status, then the pair closest in time to the anchor
                k = fu[f] - fu[i]
                best = None
                for u in range(len(g)):
                    for v in np.where((fu - fu[u]) == k)[0]:
                        if v == u or lab[u] != lab[v] or {u, v} == {i, f}:
                            continue
                        key_ = (lab[u] != lab[i], abs(fu[u] - fu[i]))
                        if best is None or key_ < best[0]:
                            best = (key_, u, v)
                if best is None:
                    continue
                _, u, v = best
                anchors.append(dict(patient=pid, view=view[i], img_a=img[i], lab_a=int(lab[i]),
                                    img_f=img[f], lab_f=int(lab[f]), img_s1=img[u], img_s=img[v],
                                    lab_s=int(lab[v]), gap_f=int(abs(k)), gap_s=int(abs(fu[v] - fu[u])),
                                    sgap_f=int(k), sgap_s=int(fu[v] - fu[u])))
                continue
            if match_gap:
                # control at the same signed time offset as the flip partner (same direction in time,
                # similar distance), so time/positioning drift is equal in both arms
                off = (fu[sm] - fu[i]) - (fu[f] - fu[i])
                s = sm[np.abs(off).argmin()]
                if abs((fu[s] - fu[i]) - (fu[f] - fu[i])) > gap_tol:
                    continue
            else:
                s = sm[np.abs(fu[sm] - fu[i]).argmin()]
            anchors.append(dict(patient=pid, view=view[i], img_a=img[i], lab_a=int(lab[i]),
                                img_f=img[f], lab_f=int(lab[f]), img_s=img[s], lab_s=int(lab[s]),
                                gap_f=int(abs(fu[f] - fu[i])), gap_s=int(abs(fu[s] - fu[i])),
                                sgap_f=int(fu[f] - fu[i]), sgap_s=int(fu[s] - fu[i])))
        if anchors:
            idx = rng.permutation(len(anchors))[:max_anchors_per_patient]
            rows += [anchors[i] for i in idx]
    if same_view and rows:   # a patient can appear in two view groups: keep the per-patient cap
        t = pd.DataFrame(rows).sample(frac=1, random_state=seed)
        rows = t.groupby("patient").head(max_anchors_per_patient).to_dict("records")
    t = pd.DataFrame(rows)
    if len(t):
        t["direction"] = np.where(t["lab_a"] == 0, "absent->present", "present->absent")
    return t


def flip_counts(df, findings=NIH_FINDINGS, same_view=True):
    """Feasibility table: per finding, patients with a within-patient flip (each direction),
    and anchors with BOTH a flip and a same-label partner."""
    out = []
    key = ["Patient ID", "View Position"] if same_view else ["Patient ID"]
    for f in findings:
        d = add_label(df, f).sort_values(key + ["Follow-up #"])
        diff = d.groupby(key)["label"].diff()
        up = d.loc[diff == 1, "Patient ID"].nunique()      # consecutive studies, same view
        down = d.loc[diff == -1, "Patient ID"].nunique()
        trip = build_anchor_triples(df, f, same_view=same_view, max_anchors_per_patient=1)
        out.append(dict(finding=f, patients_absent_to_present=up, patients_present_to_absent=down,
                        patients_with_flip_and_same_partner=len(trip),
                        of_which_anchor_absent=int((trip["lab_a"] == 0).sum()) if len(trip) else 0,
                        of_which_anchor_present=int((trip["lab_a"] == 1).sum()) if len(trip) else 0))
    return pd.DataFrame(out)


def sample_balanced(trip, n, seed=0):
    """Balance the two flip directions so a yes/no bias cannot masquerade as reliance."""
    parts = [g.sample(min(len(g), n // 2), random_state=seed) for _, g in trip.groupby("direction")]
    return pd.concat(parts).reset_index(drop=True)


def _logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def item_quantities(res, scale="prob"):
    """res needs p_a, p_f, p_s (P(yes) under anchor, flip partner, same-label partner).
    scale='prob' uses probability differences; scale='logit' uses log-odds differences
    (less compressed near 0/1, makes models with different calibration more comparable)."""
    r = res.copy()
    if scale == "logit":
        for c in ["p_a", "p_f", "p_s", "p_s1"]:
            if c in r:
                r[c] = _logit(r[c].to_numpy())
    r["d_change"] = (r["p_f"] - r["p_a"]).abs()          # sensitivity (should be large)
    ref = r["p_s1"] if "p_s1" in r else r["p_a"]          # pair control: |p(C2) - p(C1)|
    r["d_preserve"] = (r["p_s"] - ref).abs()             # nuisance sensitivity (should be small)
    r["R"] = r["d_change"] - r["d_preserve"]             # evidence-specific reliance
    r["signed_dir"] = (r["p_f"] - r["p_a"]) * (r["lab_f"] - r["lab_a"])  # >0 = moved the right way
    return r


def cluster_bootstrap(r, cols=("d_change", "d_preserve", "R", "signed_dir"), B=2000, seed=0, by="patient"):
    """Point estimates (mean of patient means) and percentile CIs resampling PATIENTS."""
    rng = np.random.default_rng(seed)
    pm = r.groupby(by)[list(cols)].mean()
    est = pm.mean()
    boots = np.empty((B, len(cols)))
    n = len(pm)
    arr = pm.to_numpy()
    for b in range(B):
        boots[b] = arr[rng.integers(0, n, n)].mean(axis=0)
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    return pd.DataFrame({"estimate": est.values, "ci_low": lo, "ci_high": hi, "n_patients": n}, index=list(cols))


def summarize(r, B=2000, seed=0):
    from scipy.stats import binomtest
    overall = cluster_bootstrap(r, B=B, seed=seed)
    by_dir = {k: cluster_bootstrap(g, B=B, seed=seed) for k, g in r.groupby("direction")}
    pm = r.groupby("patient")["R"].mean()
    nz = pm[pm != 0]
    sign_p = binomtest(int((nz > 0).sum()), len(nz), 0.5, alternative="greater").pvalue if len(nz) else np.nan
    acc = ((r["p_a"] > 0.5).astype(int) == r["lab_a"]).mean()
    return overall, by_dir, sign_p, acc


def paired_difference(r1, r2, col="R", B=4000, seed=0):
    """Difference in a per-item quantity between two models scored on the SAME anchors,
    with a patient-level paired bootstrap. Returns estimate, CI and two-sided bootstrap p."""
    rng = np.random.default_rng(seed)
    a = r1.groupby("patient")[col].mean(); b = r2.groupby("patient")[col].mean()
    a, b = a.align(b, join="inner")
    d = (a - b).to_numpy(); n = len(d)
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return dict(estimate=d.mean(), ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5),
                p_boot=min(p, 1.0), n_patients=n)


def gap_balance(t):
    """Check that the control is time-matched to the flip partner."""
    return pd.Series({"mean_gap_flip": t["gap_f"].mean(), "mean_gap_same": t["gap_s"].mean(),
                      "share_same_time_direction": (np.sign(t["sgap_f"]) == np.sign(t["sgap_s"])).mean()
                      if "sgap_f" in t else np.nan})


Writing pilot_lib.py


In [3]:
import os, glob, json, time, gc
import numpy as np, pandas as pd
from pilot_lib import *

FINDING = "Effusion"
QUESTION = "Is there a pleural effusion in this chest X-ray? Answer yes or no."
N_ANCHORS = 400
SEED = 0
OUT = "/kaggle/working"
MAX_PIXELS = 512 * 28 * 28
MODELS = [  # name, kind, dtype
    ("xrv-densenet121-res224-chex", "xrv", None),          # positive control
    ("Qwen/Qwen2.5-VL-3B-Instruct", "vlm", "float16"),     # pilot baseline
    ("Qwen/Qwen2.5-VL-7B-Instruct", "vlm", "float16"),
    ("google/medgemma-4b-it", "vlm", "float32"),           # Gemma-3 is unstable in float16 on T4
]

## 1. NIH metadata and images

In [4]:
def find_file(name, top="/kaggle/input"):
    for r, dirs, files in os.walk(top, followlinks=True):
        if name in files:
            return os.path.join(r, name)
        dirs[:] = [d for d in dirs if not d.startswith("images")]
    return None

print("Attached inputs:", os.listdir("/kaggle/input"))
meta = find_file("Data_Entry_2017.csv") or find_file("Data_Entry_2017_v2020.csv")
assert meta, "Attach the NIH Chest X-rays dataset (Add Input -> Datasets -> nih-chest-xrays/data)"
df = pd.read_csv(meta); df.columns = [c.strip() for c in df.columns]
root = os.path.dirname(meta)
img_paths = {os.path.basename(p): p for p in glob.glob(f"{root}/**/*.png", recursive=True)}
print(df.shape, "|", len(img_paths), "images")
assert img_paths, "Images not found - the dataset must be attached, not only the CSV"


Attached inputs: ['datasets']
(112120, 12) | 112120 images


## 2. Anchors with time-matched pair controls

In [5]:
trip = build_anchor_triples(df, FINDING, same_view=True, max_anchors_per_patient=1, seed=SEED, control="pair")
print(len(trip), "anchors |", trip["direction"].value_counts().to_dict())
print("gap balance:", gap_balance(trip).round(2).to_dict())
samp = sample_balanced(trip, N_ANCHORS, seed=SEED)
cols = ["img_a", "img_f", "img_s1", "img_s"]
samp = samp[samp[cols].isin(list(img_paths)).all(axis=1)].reset_index(drop=True)
print("sampled:", len(samp), "| patients:", samp["patient"].nunique(), "|", samp["direction"].value_counts().to_dict())
print("control status equals anchor status:", round((samp["lab_s"] == samp["lab_a"]).mean(), 2))
samp.to_csv(f"{OUT}/triples02_{FINDING}.csv", index=False)
imgs = pd.unique(samp[cols].to_numpy().ravel()); print(len(imgs), "unique images to score per model")
labels = pd.concat([samp[["img_a", "lab_a"]].set_axis(["img", "lab"], axis=1),
                    samp[["img_f", "lab_f"]].set_axis(["img", "lab"], axis=1),
                    samp[["img_s1", "lab_s"]].set_axis(["img", "lab"], axis=1),
                    samp[["img_s", "lab_s"]].set_axis(["img", "lab"], axis=1)]).drop_duplicates("img")

2713 anchors | {'absent->present': 2018, 'present->absent': 695}
gap balance: {'mean_gap_flip': 3.24, 'mean_gap_same': 3.24, 'share_same_time_direction': 1.0}
sampled: 400 | patients: 400 | {'absent->present': 200, 'present->absent': 200}
control status equals anchor status: 0.86
1425 unique images to score per model


## 3. Scorers

In [6]:
import torch
from PIL import Image
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret - gated models (MedGemma) will be skipped")
print("GPUs:", torch.cuda.device_count())

def make_xrv():
    import torchxrayvision as xrv, torchvision, skimage.io
    m = xrv.models.DenseNet(weights="densenet121-res224-chex").eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; m = m.to(dev)
    assert FINDING in m.targets, f"{FINDING} is not a trained output of the CheXpert weights: {m.targets}"
    idx = m.targets.index(FINDING)
    tfm = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)])
    @torch.no_grad()
    def score(path):
        img = skimage.io.imread(path)
        if img.ndim == 3:
            img = img[..., :3].mean(2)
        img = xrv.utils.normalize(img, 255 if img.max() <= 255 else 65535)
        x = torch.from_numpy(tfm(img[None])).float()[None].to(dev)
        return float(m(x)[0, idx]), 1.0      # op-point-normalised probability (0.5 = operating threshold)
    return score, m

def make_vlm(name, dtype):
    from transformers import AutoProcessor
    try:
        from transformers import AutoModelForImageTextToText as Auto
    except ImportError:
        from transformers import AutoModelForVision2Seq as Auto
    dt = getattr(torch, dtype)
    m = Auto.from_pretrained(name, torch_dtype=dt, device_map="auto", token=token).eval()
    kw = {"max_pixels": MAX_PIXELS} if "Qwen" in name else {}
    proc = AutoProcessor.from_pretrained(name, token=token, **kw)
    tok = proc.tokenizer
    ids = lambda ws: sorted({tok.encode(w, add_special_tokens=False)[0] for w in ws})
    YES, NO = ids(["Yes", "yes", " Yes", " yes"]), ids(["No", "no", " No", " no"])
    dev = next(m.parameters()).device

    def inputs(img):
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": QUESTION}]}]
        try:
            inp = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt")
        except Exception:
            msgs[0]["content"][0] = {"type": "image"}
            text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = proc(text=[text], images=[img], return_tensors="pt")
        out = {}
        for k, v in inp.items():
            if torch.is_tensor(v):
                v = v.to(dev)
                if v.is_floating_point():
                    v = v.to(dt)
            out[k] = v
        return out

    @torch.no_grad()
    def score(path):
        img = Image.open(path).convert("RGB")
        lp = torch.log_softmax(m(**inputs(img)).logits[0, -1].float(), -1)
        ly, ln = torch.logsumexp(lp[YES], 0), torch.logsumexp(lp[NO], 0)
        return torch.sigmoid(ly - ln).item(), (ly.exp() + ln.exp()).item()
    return score, m

GPUs: 2


## 4. Score every model on the same images (cached, checkpointed, failures skipped)

In [7]:
scores = {}
for name, kind, dtype in MODELS:
    tag = name.split("/")[-1]
    cache_file = f"{OUT}/scores02_{FINDING}_{tag}.json"
    cache = json.load(open(cache_file)) if os.path.exists(cache_file) else {}
    model = None
    try:
        if len(cache) < len(imgs):
            if kind == "vlm" and "medgemma" in name and token is None:
                print("skip", name, "(no HF_TOKEN)"); continue
            score, model = make_xrv() if kind == "xrv" else make_vlm(name, dtype)
            p0 = score(img_paths[imgs[0]])
            print(name, "first score:", p0)
            if not np.isfinite(p0[0]):
                print("  NaN output - set this model's dtype to float32 and re-run"); continue
            t = time.time()
            for i, im in enumerate(imgs):
                if im not in cache:
                    cache[im] = score(img_paths[im])
                if i % 100 == 0:
                    json.dump(cache, open(cache_file, "w")); print(f"  {i}/{len(imgs)} {time.time() - t:.0f}s")
            json.dump(cache, open(cache_file, "w"))
        scores[name] = cache
        print("done", name, len(cache))
    except Exception as e:
        print("FAILED", name, repr(e)[:300])
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/chex-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
xrv-densenet121-res224-chex first score: (0.3801417648792267, 1.0)
  0/1425 0s
  100/1425 8s
  200/1425 15s
  300/1425 23s
  400/1425 31s
  500/1425 38s
  600/1425 45s
  700/1425 53s
  800/1425 60s
  900/1425 68s
  1000/1425 75s
  1100/1425 83s
  1200/1425 91s
  1300/1425 98s
  1400/1425 106s
done xrv-densenet121-res224-chex 1425


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

FAILED google/medgemma-4b-it OutOfMemoryError('CUDA out of memory. Tried to allocate 12.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 2.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 15.11 MiB is reserved by PyTorc


## 5. Estimates per model

In [8]:
from sklearn.metrics import roc_auc_score
results, rows = {}, []
for name, cache in scores.items():
    res = samp.copy()
    for c in ["a", "f", "s1", "s"]:
        res[f"p_{c}"] = res[f"img_{c}"].map(lambda k: cache[k][0])
    mass = np.mean([cache[k][1] for k in imgs])
    lab = labels.set_index("img")["lab"]; p = pd.Series({k: cache[k][0] for k in lab.index})
    auc = roc_auc_score(lab.values, p[lab.index].values)
    for scale in ["prob", "logit"]:
        r = item_quantities(res, scale=scale)
        ov, by_dir, sign_p, acc = summarize(r, B=2000, seed=SEED)
        results[(name, scale)] = r
        row = dict(model=name.split("/")[-1], scale=scale, AUC=auc, yesno_mass=mass, sign_test_p=sign_p)
        for q in ["d_change", "d_preserve", "R", "signed_dir"]:
            row[q] = ov.loc[q, "estimate"]; row[q + "_lo"] = ov.loc[q, "ci_low"]; row[q + "_hi"] = ov.loc[q, "ci_high"]
        for d, v in by_dir.items():
            row["R_" + d] = v.loc["R", "estimate"]
        rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/summary02_{FINDING}.csv", index=False)
show = ["model", "scale", "AUC", "yesno_mass", "d_change", "d_preserve", "R", "R_lo", "R_hi", "signed_dir",
        "R_absent->present", "R_present->absent", "sign_test_p"]
pd.set_option("display.width", 250)
display(summary[[c for c in show if c in summary]].round(3))
for name in scores:
    results[(name, "prob")].to_csv(f"{OUT}/items02_{FINDING}_{name.split('/')[-1]}.csv", index=False)

,model,scale,AUC,yesno_mass,d_change,d_preserve,R,R_lo,R_hi,signed_dir,R_absent->present,R_present->absent,sign_test_p
0,xrv-densenet121-res224-chex,prob,0.628,1.000,0.211,0.182,0.029,0.001,0.056,0.095,0.025,0.033,0.040
1,xrv-densenet121-res224-chex,logit,0.628,1.000,1.393,1.244,0.148,-0.013,0.310,0.584,0.075,0.221,0.171
2,Qwen2.5-VL-3B-Instruct,prob,0.541,0.998,0.050,0.051,-0.001,-0.006,0.004,0.004,0.001,-0.003,0.894
3,Qwen2.5-VL-3B-Instruct,logit,0.541,0.998,0.209,0.214,-0.005,-0.026,0.017,0.018,0.005,-0.015,0.927
4,Qwen2.5-VL-7B-Instruct,prob,0.530,1.000,0.031,0.027,0.004,-0.000,0.008,-0.000,0.005,0.003,0.049
5,Qwen2.5-VL-7B-Instruct,logit,0.530,1.000,0.948,0.859,0.089,0.004,0.170,0.039,0.072,0.107,0.073


## 6. Paired model comparisons on R (same anchors, patient bootstrap, log-odds scale)

In [9]:
names = list(scores)
comp = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        d = paired_difference(results[(names[i], "logit")], results[(names[j], "logit")], col="R", seed=SEED)
        comp.append(dict(model_1=names[i].split("/")[-1], model_2=names[j].split("/")[-1], **d))
comp = pd.DataFrame(comp)
if len(comp):
    p = comp["p_boot"].to_numpy(); o = np.argsort(p); m = len(p)
    adj = np.minimum.accumulate((p[o] * m / np.arange(1, m + 1))[::-1])[::-1]
    comp["p_fdr"] = np.empty(m); comp.loc[o, "p_fdr"] = np.minimum(adj, 1)   # Benjamini-Hochberg
    comp.to_csv(f"{OUT}/paired02_{FINDING}.csv", index=False)
    display(comp.round(3))

,model_1,model_2,estimate,ci_low,ci_high,p_boot,n_patients,p_fdr
0,xrv-densenet121-res224-chex,Qwen2.5-VL-3B-Instruct,0.153,-0.010,0.312,0.070,400,0.104
1,xrv-densenet121-res224-chex,Qwen2.5-VL-7B-Instruct,0.059,-0.123,0.230,0.512,400,0.512
2,Qwen2.5-VL-3B-Instruct,Qwen2.5-VL-7B-Instruct,-0.094,-0.179,-0.007,0.035,400,0.104


## 7. Reading the results
- **Positive control first.** The CheXpert DenseNet should have AUC well above 0.5 and **R > 0 with a CI excluding 0**. If its R is near 0 despite good AUC, the pairs or the estimator are broken - stop and report.
- **AUC vs R** separates two failures: AUC ~ 0.5 means the model cannot see the finding (like Qwen-3B in the pilot); AUC > 0.5 but R ~ 0 would mean it discriminates across patients without tracking within-patient change, i.e. it relies on patient-level cues rather than the finding.
- **R_absent->present vs R_present->absent**: a big asymmetry points to an answer bias.
- Use the **log-odds rows** to compare models; probability-scale R depends on each model's calibration.
- NIH labels are text-mined, so all R values are attenuated by label noise; compare models, do not read absolute values too literally.

**Send back:** the whole `/kaggle/working` folder as a zip (`summary02_*.csv`, `paired02_*.csv`, `items02_*.csv`, `scores02_*.json`, `triples02_*.csv`), plus any FAILED / NaN messages from step 4.